In [0]:
GOLD_PATH = "/Volumes/workspace/default/logs_volume/gold"
ANOMALY_PATH = "/Volumes/workspace/default/logs_volume/anomalies"

In [0]:
# Load all three gold tables
hourly_df   = spark.read.format("delta").load(f"{GOLD_PATH}/hourly_stats")
ip_df       = spark.read.format("delta").load(f"{GOLD_PATH}/ip_behaviour")
endpoint_df = spark.read.format("delta").load(f"{GOLD_PATH}/endpoint_stats")

print(f"hourly_stats:   {hourly_df.count():,} rows")
print(f"ip_behaviour:   {ip_df.count():,} rows")
print(f"endpoint_stats: {endpoint_df.count():,} rows")

# Section 1 - Hourly Traffic Anomalies
Cases where;
  - Traffic is unusually high (spike)
  - Traffic is unusually low (drop)
  - Behaviour deviates from the normal pattern.

In [0]:
hourly_df.printSchema()
hourly_df.show(5, truncate=False)

In [0]:
from pyspark.sql.functions import (
    avg, stddev
)
# Compute the global mean and stddev across all hours
hourly_stats = hourly_df.agg(
    avg("total_requests").alias("global_mean_requests"),
    stddev("total_requests").alias("global_stddev_requests"),
    avg("error_rate").alias("global_mean_error_rate"),
    stddev("error_rate").alias("global_stddev_error_rate"),
    avg("total_bytes").alias("global_mean_bytes"),
    stddev("total_bytes").alias("global_stddev_bytes")
).collect()[0]

global_mean_req      = hourly_stats["global_mean_requests"]
global_stddev_req    = hourly_stats["global_stddev_requests"]
global_mean_err      = hourly_stats["global_mean_error_rate"]
global_stddev_err    = hourly_stats["global_stddev_error_rate"]
global_mean_bytes    = hourly_stats["global_mean_bytes"]
global_stddev_bytes  = hourly_stats["global_stddev_bytes"]

print(f"Baseline — mean requests/hr: {global_mean_req:.0f}  stddev: {global_stddev_req:.0f}")
print(f"Baseline — mean error rate:  {global_mean_err:.4f}  stddev: {global_stddev_err:.4f}")

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import lag, col, lit, when

time_window = Window.orderBy("hour_bucket")

hourly_enriched = (hourly_df
    # Previous hour values
    .withColumn("prev_requests", lag("total_requests", 1).over(time_window))
    .withColumn("prev_error_rate", lag("error_rate", 1).over(time_window))
    .withColumn("prev_bytes", lag("total_bytes", 1).over(time_window))

    # Hour-over-hour change %
    .withColumn(
        "request_change_pct",
        when(
            col("prev_requests").isNull() | (col("prev_requests") == 0),
            lit(0)
        ).otherwise(
            (col("total_requests") - col("prev_requests")) /
            col("prev_requests") * 100
        )
    )

    .withColumn(
        "error_rate_change_pct",
        when(
            col("prev_error_rate").isNull() | (col("prev_error_rate") == 0),
            lit(0)
        ).otherwise(
            (col("error_rate") - col("prev_error_rate")) /
            (col("prev_error_rate") + lit(1e-4)) * 100
        )
    )

    # Z-scores (safe against zero stddev)
    .withColumn(
        "request_z_score",
        when(lit(global_stddev_req) == 0, lit(0))
        .otherwise(
            (col("total_requests") - lit(global_mean_req)) /
            lit(global_stddev_req)
        )
    )

    .withColumn(
        "error_rate_z_score",
        when(lit(global_stddev_err) == 0, lit(0))
        .otherwise(
            (col("error_rate") - lit(global_mean_err)) /
            lit(global_stddev_err)
        )
    )

    .withColumn(
        "bytes_z_score",
        when(lit(global_stddev_bytes) == 0, lit(0))
        .otherwise(
            (col("total_bytes") - lit(global_mean_bytes)) /
            lit(global_stddev_bytes)
        )
    )
)

hourly_enriched.show(5)

In [0]:
SPIKE_Z       = 2.5   # standard deviations above mean = suspicious
ERROR_SURGE_Z = 2.5
QUIET_Z       = -2.5  # unusually low traffic = also suspicious (outage?)
HOH_SPIKE_PCT = 100   # 100% jump hour-over-hour = suspicious

hourly_anomalies = (hourly_enriched
    .withColumn("anomaly_type",
        when(col("request_z_score") > SPIKE_Z,         "TRAFFIC_SPIKE")
        .when(col("request_z_score") < QUIET_Z,        "UNUSUAL_QUIET")
        .when(col("error_rate_z_score") > ERROR_SURGE_Z, "ERROR_SURGE")
        .when(col("request_change_pct") > HOH_SPIKE_PCT, "SUDDEN_HOH_JUMP")
        .otherwise(None)
    )
    .filter(col("anomaly_type").isNotNull())
    .select(
        lit("HOURLY").alias("anomaly_source"),
        col("hour_bucket").alias("detected_at"),
        lit(None).cast("string").alias("host"),
        lit(None).cast("string").alias("endpoint"),
        col("anomaly_type"),
        col("request_z_score").alias("z_score"),
        col("total_requests").alias("metric_value"),
        col("request_change_pct").alias("change_pct"),
        col("rolling_3hr_avg_requests").alias("baseline_value")
    )
)

print(f"Hourly anomalies detected: {hourly_anomalies.count()}")
hourly_anomalies.show(10, truncate=False)

#Section 2 - IP Behaviour Anomalies.

In [0]:
# Compute IP Baselines
ip_stats = ip_df.agg(
    avg("total_requests").alias("mean_requests"),
    stddev("total_requests").alias("stddev_requests"),
    avg("error_rate_pct").alias("mean_error_rate"),
    stddev("error_rate_pct").alias("stddev_error_rate"),
    avg("unique_endpoints_hit").alias("mean_endpoints"),
    stddev("unique_endpoints_hit").alias("stddev_endpoints"),
    avg("total_bytes_transferred").alias("mean_bytes"),
    stddev("total_bytes_transferred").alias("stddev_bytes")
).collect()[0]

print("IP Baselines:")
for k, v in ip_stats.asDict().items():
    print(f"  {k}: {v:.2f}")

In [0]:
# Score every IP across 4 dimension
from pyspark.sql.functions import round

ip_scored = (ip_df
    # Z-score: volume (high = potential scraper/bot)
    .withColumn("volume_z",
        round((col("total_requests") - lit(ip_stats["mean_requests"]))
              / lit(ip_stats["stddev_requests"]), 2))

    # Z-score: error rate (high = scanner probing bad endpoints)
    .withColumn("error_z",
        round((col("error_rate_pct") - lit(ip_stats["mean_error_rate"]))
              / lit(ip_stats["stddev_error_rate"]), 2))

    # Z-score: endpoint breadth (high = crawling many endpoints)
    .withColumn("breadth_z",
        round((col("unique_endpoints_hit") - lit(ip_stats["mean_endpoints"]))
              / lit(ip_stats["stddev_endpoints"]), 2))

    # Z-score: data transferred (high = bulk downloader)
    .withColumn("bytes_z",
        round((col("total_bytes_transferred") - lit(ip_stats["mean_bytes"]))
              / lit(ip_stats["stddev_bytes"]), 2))

    # Composite suspicion score: weighted sum of all signals
    .withColumn("suspicion_score",
        round(
            (col("volume_z")  * 0.3) +
            (col("error_z")   * 0.3) +
            (col("breadth_z") * 0.25) +
            (col("bytes_z")   * 0.15),
        2))
)

ip_scored.show(5, truncate=False)

In [0]:
# Classify IP Anomaly type
ip_anomalies = (ip_scored
    .withColumn("anomaly_type",
        when(
            (col("volume_z") > 3) & (col("breadth_z") > 3),   "BOT_SCRAPER"
        ).when(
            (col("error_z") > 3) & (col("volume_z") > 1),     "ENDPOINT_SCANNER"
        ).when(
            (col("bytes_z") > 3) & (col("volume_z") < 1),     "BULK_DOWNLOADER"
        ).when(
            col("error_rate_pct") > 20,                        "HIGH_ERROR_IP"
        ).when(
            col("suspicion_score") > 2.5,                      "GENERALLY_SUSPICIOUS"
        ).otherwise(None)
    )
    .filter(col("anomaly_type").isNotNull())
    .select(
        lit("IP").alias("anomaly_source"),
        col("first_seen").alias("detected_at"),
        col("host"),
        lit(None).cast("string").alias("endpoint"),
        col("anomaly_type"),
        col("suspicion_score").alias("z_score"),
        col("total_requests").alias("metric_value"),
        col("error_rate_pct").alias("change_pct"),
        col("total_requests").cast("double").alias("baseline_value")
    )
)

print(f"IP anomalies detected: {ip_anomalies.count()}")
ip_anomalies.groupBy("anomaly_type").count().show()

# Section 3 - Endpoint Anomalies

In [0]:
# Score Endpoint
ep_stats = endpoint_df.agg(
    avg("not_found_count").alias("mean_404"),
    stddev("not_found_count").alias("stddev_404"),
    avg("avg_bytes").alias("mean_bytes"),
    stddev("avg_bytes").alias("stddev_bytes"),
    avg("total_hits").alias("mean_hits"),
    stddev("total_hits").alias("stddev_hits")
).collect()[0]

endpoint_scored = (endpoint_df
    .withColumn("not_found_rate",
        round(col("not_found_count") / col("total_hits") * 100, 2))

    .withColumn("404_z",
        round((col("not_found_count") - lit(ep_stats["mean_404"]))
              / lit(ep_stats["stddev_404"]), 2))

    .withColumn("bytes_z",
        round((col("avg_bytes") - lit(ep_stats["mean_bytes"]))
              / lit(ep_stats["stddev_bytes"]), 2))

    .withColumn("hits_z",
        round((col("total_hits") - lit(ep_stats["mean_hits"]))
              / lit(ep_stats["stddev_hits"]), 2))
)

endpoint_anomalies = (endpoint_scored
    .withColumn("anomaly_type",
        when(
            col("not_found_rate") > 50,                        "MOSTLY_BROKEN_ENDPOINT"
        ).when(
            (col("404_z") > 3) & (col("hits_z") > 2),         "ATTACKED_ENDPOINT"
        ).when(
            col("bytes_z") > 3,                                "UNUSUALLY_HEAVY_PAYLOAD"
        ).when(
            (col("hits_z") > 4) & (col("not_found_rate") < 1), "HOTSPOT_ENDPOINT"
        ).otherwise(None)
    )
    .filter(col("anomaly_type").isNotNull())
    .select(
        lit("ENDPOINT").alias("anomaly_source"),
        lit(None).cast("timestamp").alias("detected_at"),
        lit(None).cast("string").alias("host"),
        col("endpoint"),
        col("anomaly_type"),
        col("hits_z").alias("z_score"),
        col("total_hits").alias("metric_value"),
        col("not_found_rate").alias("change_pct"),
        col("avg_bytes").alias("baseline_value")
    )
)

print(f"Endpoint anomalies detected: {endpoint_anomalies.count()}")
endpoint_anomalies.groupBy("anomaly_type").count().show()

# Severity Report

In [0]:
# Union all anomalies into one master table
all_anomalies = (hourly_anomalies
    .union(ip_anomalies)
    .union(endpoint_anomalies)
)

print(f"Total anomalies across all sources: {all_anomalies.count()}")
all_anomalies.groupBy("anomaly_source", "anomaly_type").count().orderBy("anomaly_source").show(30)

In [0]:
final_anomalies = (all_anomalies
    .withColumn("severity",
        when(col("z_score") >= 6,  "CRITICAL")
        .when(col("z_score") >= 4, "HIGH")
        .when(col("z_score") >= 2.5, "MEDIUM")
        .otherwise("LOW")
    )
    .orderBy("severity", col("z_score").desc())
)

# final_anomalies.show(5)

(final_anomalies
    .write
    .format("delta")
    .mode("overwrite")
    .save(ANOMALY_PATH))

print("Anomaly table written successfully.")

In [0]:
print("=" * 50)
print("        ANOMALY DETECTION SUMMARY")
print("=" * 50)

summary = (final_anomalies
    .groupBy("anomaly_source", "anomaly_type", "severity")
    .count()
    .orderBy("severity", "anomaly_source"))

summary.show(50, truncate=False)

print("\nBy severity:")
final_anomalies.groupBy("severity").count().orderBy("severity").show()

print("\nCRITICAL anomalies:")
(final_anomalies
    .filter(col("severity") == "CRITICAL")
    .select("anomaly_source", "anomaly_type", "host", "endpoint", "z_score", "metric_value")
    .show(20, truncate=False))